# AutoKeras robusto

Pipeline robusto para o cenário binário do CICIoT2023 no Google Colab.

**Framework:** `autokeras`

Este notebook segue a mesma estrutura lógica do pipeline Auto-PyTorch: leitura em chunks, remoção de `Merged*.csv`, criação de rótulos, limpeza, `float32`, divisão 80/20 estratificada, treinamento, avaliação externa e salvamento do modelo individual.

In [ ]:
# ============================================================
# 00. IDENTIFICAÇÃO DO FRAMEWORK
# ============================================================
FRAMEWORK_NAME = 'autokeras'

In [ ]:
# ============================================================
# INSTALAÇÃO / VALIDAÇÃO - AUTOKERAS
# ============================================================
# Finalidade:
# - Instalar AutoKeras e TensorFlow;
# - Usar StructuredDataClassifier para dados tabulares.
# ============================================================
!pip -q install -U autokeras tensorflow pandas numpy matplotlib scikit-learn joblib

In [ ]:
# ============================================================
# CONFIGURAÇÕES GERAIS DO PIPELINE BINÁRIO - CICIoT2023
# ============================================================
# Finalidade:
# - Definir caminhos do Google Drive;
# - Definir leitura em chunks;
# - Ignorar arquivos Merged*.csv;
# - Criar os rótulos label_original, label_binary, label_grouped e label_multiclass;
# - Preparar X/y do cenário binário;
# - Usar split 80% treino e 20% teste;
# - Manter stratify=True e random_state=42;
# - Usar float32 para reduzir consumo de memória;
# - Reservar o teste externo apenas para avaliação final.
# ============================================================

from pathlib import Path
import os
import re
import gc
import json
import time
import shutil
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Aviso: Google Drive pode já estar montado ou o código não está no Colab:', exc)

# ----------------------------
# Caminhos principais
# ----------------------------
DATASET_DIR = Path('/content/drive/MyDrive/Dataset/CSV')
RESULTS_BASE = Path('/content/drive/MyDrive/Dataset/Resultados/AutoML_Comparativo')

# FRAMEWORK_NAME é definido em cada notebook individual.
RESULTS_DIR = RESULTS_BASE / FRAMEWORK_NAME / 'cenario_01_binario'
MODEL_DIR = RESULTS_DIR / f'modelo_{FRAMEWORK_NAME}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Parâmetros de robustez
# ----------------------------
CHUNKSIZE = 200_000
MAX_FILES = None
MAX_ROWS_PER_CLASS = None  # None = usa o máximo disponível. Para teste rápido: 50_000, 100_000, 500_000.
TEST_SIZE = 0.20
RANDOM_STATE = 42
MEMORY_LIMIT_MB = 150_000
TOTAL_WALLTIME_LIMIT = 10_800      # 3 horas
FUNC_EVAL_TIME_LIMIT_SECS = 1_200  # 20 minutos por tentativa/modelo
N_JOBS = -1
USE_GPU = False

LABEL_ORIGINAL = 'label_original'
LABEL_BINARY = 'label_binary'
LABEL_GROUPED = 'label_grouped'
LABEL_MULTICLASS = 'label_multiclass'
TARGET_COL = LABEL_BINARY

CLASSE_BENIGNA = 'Benign'
CLASSE_MALICIOSA = 'Malicious'

LABEL_CANDIDATES = [
    'label', 'Label', 'Attack', 'attack', 'Class', 'class',
    'category', 'Category', 'label_multiclass', 'label_original'
]

print('=' * 70)
print(f'PIPELINE BINÁRIO CICIoT2023 - {FRAMEWORK_NAME}')
print('=' * 70)
print('Dataset:', DATASET_DIR)
print('Resultados:', RESULTS_DIR)
print('Modelo:', MODEL_DIR)
print('CHUNKSIZE:', CHUNKSIZE)
print('MAX_FILES:', MAX_FILES)
print('MAX_ROWS_PER_CLASS:', MAX_ROWS_PER_CLASS)
print('MEMORY_LIMIT_MB:', MEMORY_LIMIT_MB)

In [ ]:
# ============================================================
# FUNÇÕES COMUNS DE CARREGAMENTO, ROTULAGEM E AVALIAÇÃO
# ============================================================
# Finalidade:
# - Carregar os CSVs em chunks de 200.000 linhas;
# - Ignorar arquivos Merged*.csv;
# - Extrair label_original do CSV ou do nome do arquivo;
# - Criar label_binary, label_grouped e label_multiclass;
# - Remover colunas de vazamento: labels e source_file;
# - Tratar NaN/Inf;
# - Converter features para float32;
# - Separar treino/teste de forma estratificada;
# - Salvar métricas, matriz de confusão, classification report e resumo JSON.
# ============================================================

def normalizar_rotulo_arquivo(path: Path) -> str:
    """Extrai a classe original a partir do nome do arquivo CSV."""
    nome = path.name
    nome = re.sub(r'\.pcap\.csv$', '', nome, flags=re.IGNORECASE)
    nome = re.sub(r'\.csv$', '', nome, flags=re.IGNORECASE)
    # Remove sufixos numéricos de arquivos particionados: BenignTraffic1 -> BenignTraffic.
    nome = re.sub(r'\d+$', '', nome)
    return nome


def detectar_coluna_rotulo(df: pd.DataFrame):
    """Detecta uma coluna de rótulo quando ela existe no CSV."""
    for col in LABEL_CANDIDATES:
        if col in df.columns:
            return col
    return None


def converter_para_binario(label) -> str:
    """Converte a classe original em Benign ou Malicious."""
    texto = str(label).lower()
    if 'benign' in texto:
        return CLASSE_BENIGNA
    return CLASSE_MALICIOSA


def converter_para_agrupado(label) -> str:
    """Agrupa as classes originais em 8 classes principais."""
    texto = str(label)
    baixo = texto.lower()
    if 'benign' in baixo:
        return 'Benign'
    if texto.startswith('DDoS-'):
        return 'DDoS'
    if texto.startswith('DoS-'):
        return 'DoS'
    if texto.startswith('Mirai-'):
        return 'Mirai'
    if texto.startswith('Recon-'):
        return 'Recon'
    if texto in ['DNS_Spoofing']:
        return 'Spoofing'
    if texto in ['DictionaryBruteForce']:
        return 'BruteForce'
    if texto in ['BrowserHijacking', 'CommandInjection', 'Backdoor_Malware']:
        return 'Web'
    return 'Outros'


def listar_csvs():
    """Lista arquivos CSV e remove os arquivos consolidados Merged*.csv."""
    csv_files = sorted(DATASET_DIR.glob('*.csv'))
    total_original = len(csv_files)
    merged = [f for f in csv_files if f.name.lower().startswith('merged')]
    csv_files = [f for f in csv_files if not f.name.lower().startswith('merged')]
    if MAX_FILES is not None:
        csv_files = csv_files[:MAX_FILES]
    print('Total CSV original:', total_original)
    print('Total CSV Merged ignorados:', len(merged))
    print('Total CSV usados:', len(csv_files))
    return csv_files


def limitar_por_classe(chunk: pd.DataFrame, contador: dict) -> pd.DataFrame:
    """Aplica limite opcional MAX_ROWS_PER_CLASS por label_original."""
    if MAX_ROWS_PER_CLASS is None:
        return chunk
    partes = []
    for classe, sub in chunk.groupby(LABEL_ORIGINAL, sort=False):
        atual = contador.get(classe, 0)
        restante = MAX_ROWS_PER_CLASS - atual
        if restante <= 0:
            continue
        selecionado = sub.head(restante)
        contador[classe] = atual + len(selecionado)
        partes.append(selecionado)
    if partes:
        return pd.concat(partes, ignore_index=True)
    return pd.DataFrame()


def carregar_dataset() -> pd.DataFrame:
    """Carrega o CICIoT2023 em chunks, criando label_original e source_file."""
    csv_files = listar_csvs()
    partes = []
    contador = {}
    total = 0
    inicio = time.time()

    for i, arquivo in enumerate(csv_files, start=1):
        print(f'[{i}/{len(csv_files)}] Lendo: {arquivo.name}')
        rotulo_arquivo = normalizar_rotulo_arquivo(arquivo)
        try:
            for j, chunk in enumerate(pd.read_csv(arquivo, chunksize=CHUNKSIZE, low_memory=False), start=1):
                coluna_rotulo = detectar_coluna_rotulo(chunk)
                if coluna_rotulo is not None:
                    chunk[LABEL_ORIGINAL] = chunk[coluna_rotulo].astype(str)
                else:
                    chunk[LABEL_ORIGINAL] = rotulo_arquivo
                chunk['source_file'] = arquivo.name
                chunk = limitar_por_classe(chunk, contador)
                if len(chunk) == 0:
                    continue
                partes.append(chunk)
                total += len(chunk)
                print(f'  Chunk {j}: {len(chunk)} linhas | Total acumulado: {total}')
        except Exception as exc:
            print(f'  ERRO ao ler {arquivo.name}: {exc}')

    if not partes:
        raise RuntimeError('Nenhuma linha foi carregada. Verifique DATASET_DIR e os CSVs.')

    print('Concatenando partes carregadas...')
    df = pd.concat(partes, ignore_index=True)
    del partes
    gc.collect()
    print('Dataset carregado.')
    print('Shape:', df.shape)
    print(f'Tempo de carregamento: {time.time() - inicio:.2f} segundos')
    return df


def criar_rotulos(df: pd.DataFrame) -> pd.DataFrame:
    """Cria os três cenários de rótulo a partir de label_original."""
    df[LABEL_ORIGINAL] = df[LABEL_ORIGINAL].astype(str)
    df[LABEL_BINARY] = df[LABEL_ORIGINAL].apply(converter_para_binario)
    df[LABEL_GROUPED] = df[LABEL_ORIGINAL].apply(converter_para_agrupado)
    df[LABEL_MULTICLASS] = df[LABEL_ORIGINAL].astype(str)

    diagnostico = df[LABEL_ORIGINAL].value_counts().reset_index()
    diagnostico.columns = [LABEL_ORIGINAL, 'quantidade']
    diagnostico.to_csv(RESULTS_DIR / f'diagnostico_rotulo_original_{FRAMEWORK_NAME}.csv', index=False)

    print('Rótulos criados.')
    print('Distribuição binária:')
    print(df[LABEL_BINARY].value_counts())
    return df


def preparar_X_y(df: pd.DataFrame):
    """Prepara X e y, removendo colunas de vazamento e convertendo features para float32."""
    colunas_vazamento = set([
        LABEL_ORIGINAL, LABEL_BINARY, LABEL_GROUPED, LABEL_MULTICLASS,
        'source_file', 'label', 'Label', 'Attack', 'attack', 'Class', 'class', 'category', 'Category'
    ])
    remover = [c for c in df.columns if c in colunas_vazamento]
    X = df.drop(columns=remover, errors='ignore')
    y = df[TARGET_COL].astype(str)

    # Mantém apenas colunas numéricas ou convertíveis para número.
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.dropna(axis=1, how='all')

    medianas = X.median(numeric_only=True)
    X = X.fillna(medianas)
    X = X.astype(np.float32)

    print('Features preparadas.')
    print('X:', X.shape)
    print('y:', y.shape)
    print('Memória X total:', round(X.memory_usage(deep=True).sum() / (1024**3), 2), 'GB')
    return X, y


def dividir_treino_teste(X, y):
    """Divide em 80% treino e 20% teste, preservando proporções das classes."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    X_train = X_train.astype(np.float32).reset_index(drop=True)
    X_test = X_test.astype(np.float32).reset_index(drop=True)
    y_train = pd.Series(y_train).astype(str).reset_index(drop=True)
    y_test = pd.Series(y_test).astype(str).reset_index(drop=True)

    print('Divisão treino/teste:')
    print('X_train:', X_train.shape)
    print('X_test:', X_test.shape)
    print('y_train:', y_train.shape)
    print('y_test:', y_test.shape)
    print('Memória X_train:', round(X_train.memory_usage(deep=True).sum() / (1024**3), 2), 'GB')
    print('Memória X_test:', round(X_test.memory_usage(deep=True).sum() / (1024**3), 2), 'GB')
    return X_train, X_test, y_train, y_test


def salvar_dataframe_png(df: pd.DataFrame, caminho_png: Path, titulo: str):
    """Salva uma tabela pandas como imagem PNG sem usar seaborn."""
    if df is None or len(df) == 0:
        return
    fig_h = max(2, min(0.45 * (len(df) + 1), 30))
    fig_w = max(8, min(2.2 * len(df.columns), 30))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    ax.set_title(titulo)
    tabela = ax.table(
        cellText=df.round(6).astype(str).values,
        colLabels=df.columns,
        loc='center',
        cellLoc='center'
    )
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(8)
    tabela.scale(1, 1.2)
    plt.tight_layout()
    fig.savefig(caminho_png, dpi=200, bbox_inches='tight')
    plt.close(fig)


def salvar_metricas_e_graficos(y_test, y_pred, nome_modelo: str, extra=None):
    """Salva métricas, matriz de confusão, classification report e gráfico de métricas."""
    extra = extra or {}
    labels = sorted(pd.Series(y_test).astype(str).unique().tolist())

    metricas = {
        'Framework': FRAMEWORK_NAME,
        'Modelo': nome_modelo,
        'Cenario': 'binario',
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'Precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'Total_teste': len(y_test),
        'Memory_limit_MB': MEMORY_LIMIT_MB,
        'Walltime_limit_s': TOTAL_WALLTIME_LIMIT,
        'Eval_time_limit_s': FUNC_EVAL_TIME_LIMIT_SECS,
    }
    metricas.update(extra)
    df_metricas = pd.DataFrame([metricas])
    df_metricas.to_csv(RESULTS_DIR / f'metricas_{FRAMEWORK_NAME}.csv', index=False)
    salvar_dataframe_png(df_metricas, RESULTS_DIR / f'grafico_metricas_{FRAMEWORK_NAME}.png', f'Métricas - {FRAMEWORK_NAME}')

    cm = confusion_matrix(y_test, y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)
    df_cm.to_csv(RESULTS_DIR / f'matriz_confusao_{FRAMEWORK_NAME}.csv')

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_title(f'Matriz de Confusão - {FRAMEWORK_NAME}')
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f'matriz_confusao_{FRAMEWORK_NAME}.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    df_report = pd.DataFrame(report).T.reset_index().rename(columns={'index': 'classe'})
    df_report.to_csv(RESULTS_DIR / f'classification_report_{FRAMEWORK_NAME}.csv', index=False)
    salvar_dataframe_png(df_report, RESULTS_DIR / f'classification_report_{FRAMEWORK_NAME}.png', f'Classification Report - {FRAMEWORK_NAME}')

    resumo = {
        'framework': FRAMEWORK_NAME,
        'modelo': nome_modelo,
        'cenario': 'binario',
        'target': TARGET_COL,
        'dataset_dir': str(DATASET_DIR),
        'results_dir': str(RESULTS_DIR),
        'model_dir': str(MODEL_DIR),
        'chunksize': CHUNKSIZE,
        'max_files': MAX_FILES,
        'max_rows_per_class': MAX_ROWS_PER_CLASS,
        'test_size': TEST_SIZE,
        'random_state': RANDOM_STATE,
        'memory_limit_mb': MEMORY_LIMIT_MB,
        'total_walltime_limit': TOTAL_WALLTIME_LIMIT,
        'func_eval_time_limit_secs': FUNC_EVAL_TIME_LIMIT_SECS,
        'use_gpu': USE_GPU,
        'metricas': metricas,
    }
    with open(RESULTS_DIR / f'resumo_experimento_{FRAMEWORK_NAME}.json', 'w', encoding='utf-8') as f:
        json.dump(resumo, f, ensure_ascii=False, indent=2)

    print('Métricas salvas em:', RESULTS_DIR)
    display(df_metricas)
    return df_metricas


def carregar_e_preparar_dados():
    """Executa todo o pré-processamento comum do cenário binário."""
    df = carregar_dataset()
    df = criar_rotulos(df)
    X, y = preparar_X_y(df)
    del df
    gc.collect()
    return dividir_treino_teste(X, y)

In [ ]:
# ============================================================
# TREINAMENTO - AUTOKERAS
# ============================================================
# Finalidade:
# - Treinar AutoKeras StructuredDataClassifier;
# - Usar validação interna via validation_split;
# - Salvar modelo Keras exportado individualmente;
# - Avaliar no teste externo com Macro-F1.
# ============================================================

import joblib
import tensorflow as tf
import autokeras as ak
from sklearn.preprocessing import LabelEncoder

if not USE_GPU:
    try:
        tf.config.set_visible_devices([], 'GPU')
    except Exception:
        pass

X_train, X_test, y_train, y_test = carregar_e_preparar_dados()

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
joblib.dump(le, MODEL_DIR / 'label_encoder_autokeras.joblib')

clf = ak.StructuredDataClassifier(
    max_trials=50,
    objective='val_accuracy',
    overwrite=True,
    directory=str(MODEL_DIR),
    project_name='modelo_autokeras',
)

inicio = time.time()
clf.fit(
    X_train,
    y_train_enc,
    validation_split=0.2,
    epochs=50,
    batch_size=8192,
    verbose=1,
)
tempo_treino = time.time() - inicio

modelo_exportado = clf.export_model()
caminho_modelo = MODEL_DIR / 'modelo_autokeras.keras'
modelo_exportado.save(caminho_modelo)
print('Modelo salvo em:', caminho_modelo)

inicio_pred = time.time()
y_pred_raw = clf.predict(X_test, batch_size=8192).reshape(-1)
tempo_predicao = time.time() - inicio_pred

y_pred_enc = y_pred_raw.astype(int)
y_pred = le.inverse_transform(y_pred_enc)

salvar_metricas_e_graficos(
    y_test,
    y_pred,
    nome_modelo='AutoKeras_StructuredDataClassifier',
    extra={'tempo_treino_s': tempo_treino, 'tempo_predicao_s': tempo_predicao}
)

print('Fim do pipeline AutoKeras.')